# 19 - Healthcare: One Person as Four Concepts

The same individual is a PATIENT (clinical, treatment basis), a
MEMBER (coverage and eligibility), a CONSUMER (marketing, a
different consent regime), and a SUBJECT (research, protocol
governed). One record, four governing rule sets — an agent that
treats them as one identity applies the wrong permissions.

Identity is never a caller claim here: each context is a
separate MCP token whose `bound_role` was verified at issuance.

| Verified role | Table read | Clinical columns | Purpose lane |
| --- | --- | --- | --- |
| `clinical` | allow | visible | `care.treatment` |
| `member_services` | review | masked | `care.eligibility` |
| `research` | review | — | de-identified only |
| `marketing` | review | — | denied without opt-in |

Field-level specifics are illustrative, not clinical guidance.


In [ ]:
from pathlib import Path
import json
import os
import sys

import pandas as pd

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from common import get_client

mode = os.getenv("METATATE_EXAMPLES_MODE", "offline")
if mode == "live" and not os.getenv("METATATE_MCP_URL"):
    print("Live mode needs a Metatate endpoint. Fastest path (about 5 minutes):")
    print("  1. Create a free account: https://app.getmetatate.com/sign-up?ref=examples")
    print("  2. Workspace dashboard: 'Load the demo' banner -> 'Load the Customer 360 demo'")
    print("  3. MCP Tools -> Tokens: issue a token; Connect tab has your endpoint URL")
    print("  4. export METATATE_MCP_URL=... METATATE_SAAS_MCP_TOKEN=...")
    print("     (full steps: docs/live-mode-saas.md)")

client = get_client()
print(f"Metatate examples mode: {mode}")


PRODUCT_DATABASE_TABLES = {"product_usage_events", "support_tickets", "ml_feature_store"}


def asset(table, column=None, schema="public", database=None):
    resolved_database = database or (
        "product" if table in PRODUCT_DATABASE_TABLES else "master"
    )
    ref = {"database": resolved_database, "schema": schema, "table": table}
    if column:
        ref["column"] = column
    return ref


def answer_label(answer):
    state = answer.get("state")
    if state and state != "answered":
        return state
    return answer.get("decision") or answer.get("verdict") or "unknown"


def print_answer(answer):
    print(f"state:    {answer.get('state')}")
    if "decision" in answer:
        print(f"decision: {answer['decision']}")
    if "verdict" in answer:
        print(f"verdict:  {answer['verdict']}")
    if answer.get("reason"):
        print(f"reason:   {answer['reason']}")
    for condition in answer.get("conditions") or []:
        print(f"condition [{condition.get('kind')}]: {condition.get('requirement')}")
    for prohibition in answer.get("prohibitions") or []:
        print(f"prohibition: {prohibition.get('detail')}")
    for obligation in answer.get("obligations") or []:
        print(f"obligation [{obligation.get('type')}]: {obligation.get('target')}")
    if "can_proceed_now" in answer:
        print(f"can_proceed_now: {answer['can_proceed_now']}")


In [ ]:
# Four verified identities — four credentials. Offline mode
# replays the per-role recordings; live mode uses four separate
# role-bound tokens.
clinical = get_client(token_env="METATATE_SAAS_MCP_CLINICAL_TOKEN")
member_services = get_client(token_env="METATATE_SAAS_MCP_MEMBER_SERVICES_TOKEN")
research = get_client(token_env="METATATE_SAAS_MCP_RESEARCH_TOKEN")
marketing = get_client(token_env="METATATE_SAAS_MCP_MARKETING_TOKEN")


## 1. The table-grain role gate


In [ ]:
read_cases = [
    ("clinical", clinical, "read the full member record on a treatment basis"),
    ("member_services", member_services, "read the full member record for coverage work"),
    ("research", research, "read member records for a research cohort"),
    ("marketing", marketing, "read member records for outreach targeting"),
]

for label, role_client, use in read_cases:
    answer = role_client.authorize_use(
        asset("member_records", database="care"),
        use=use,
        scenario_key="access.read",
    )
    print(f"{label:16} -> {answer_label(answer)}")


Only the clinical role reads the full record at table grain — a
treatment basis. Every other identity fails closed to review
here: coverage work reads through the eligibility PURPOSE lane,
never a table-wide grant.


## 2. The column-grain contrast


In [ ]:
notes_clinical = clinical.authorize_use(
    asset("member_records", "clinical_notes", database="care"),
    use="display clinician notes during treatment",
    scenario_key="masking.display",
)
notes_member = member_services.authorize_use(
    asset("member_records", "clinical_notes", database="care"),
    use="display the member record in a coverage tool",
    scenario_key="masking.display",
)
print("clinical        ->", answer_label(notes_clinical))
print("member_services ->", answer_label(notes_member))


## 3. The member context reads through its purpose


In [ ]:
eligibility = member_services.authorize_use(
    asset("member_records", database="care"),
    use="check coverage eligibility for a claim",
    scenario_key="purpose.allowed_use",
    purpose_key="care.eligibility",
)
eligibility_consent = member_services.authorize_use(
    asset("member_records", database="care"),
    use="check coverage eligibility for a claim",
    scenario_key="consent.required",
    purpose_key="care.eligibility",
)
condition = next(
    (c for c in eligibility_consent.get("conditions", [])
     if c.get("kind") == "consent_required"),
    {},
)
projection = condition.get("projection") or {}
print("eligibility purpose ->", answer_label(eligibility))
print("eligibility consent ->", answer_label(eligibility_consent),
      "verify", projection.get("basis_column", "?"))


## 4. The subject context: de-identified, protocol-gated


In [ ]:
anonymize = research.authorize_use(
    asset("member_records", database="care"),
    use="analyze member outcomes for a research study",
    scenario_key="protection.anonymization",
    purpose_key="research.general",
)
research_consent = research.authorize_use(
    asset("member_records", database="care"),
    use="analyze member outcomes for a research study",
    scenario_key="consent.required",
    purpose_key="research.general",
)
training = research.authorize_use(
    asset("member_records", database="care"),
    use="train a model on member records",
    scenario_key="ai.training",
)
print("research use      ->", answer_label(anonymize))
print("research consent  ->", answer_label(research_consent))
print("model training    ->", answer_label(training))


Research is conditional twice over: anonymize first, and verify
the recorded research authorization. "Only under an active
protocol" is the steward-granted exception lane
(`request_access` -> approval -> retry with the cited
exception), which notebook 09 walks end to end — never a policy
switch an agent can flip.


## 5. The consumer context


In [ ]:
outreach = marketing.authorize_use(
    asset("member_records", database="care"),
    use="market wellness products to members",
    scenario_key="purpose.prohibited_use",
    purpose_key="marketing.advertising",
)
outreach_consent = marketing.authorize_use(
    asset("member_records", database="care"),
    use="market wellness products to members",
    scenario_key="consent.required",
    purpose_key="marketing.advertising",
)
treatment_consent = clinical.authorize_use(
    asset("member_records", database="care"),
    use="treat the member in a clinical encounter",
    scenario_key="consent.required",
    purpose_key="care.treatment",
)
print("marketing use      ->", answer_label(outreach))
print("marketing consent  ->", answer_label(outreach_consent))
print("treatment consent  ->", treatment_consent["state"], treatment_consent["reason_code"])


Marketing is denied on its purpose lane, and even its consent
question is only conditional on the recorded opt-in. Treatment
deliberately has NO consent rule — asking the consent question
for a purpose no rule names fails closed to review rather than
inventing an answer.


## 6. The role flips the SQL verdict


In [ ]:
NOTES_SQL = "SELECT member_id, clinical_notes FROM care.public.member_records"

sql_clinical = clinical.validate_query_context(
    NOTES_SQL,
    scenario_key="purpose.allowed_use",
    default_database="master", default_schema="public",
    purpose_key="care.treatment",
)
sql_member = member_services.validate_query_context(
    NOTES_SQL,
    scenario_key="purpose.allowed_use",
    default_database="master", default_schema="public",
    purpose_key="care.treatment",
)
print("clinical        ->", sql_clinical.get("verdict"), "/", sql_clinical.get("state"))
print("member_services ->", sql_member.get("verdict"), "/", sql_member.get("state"))


## 7. The receipt


In [ ]:
receipt = research.explain_why(
    authorization_id=anonymize["authorization_id"],
)
print("decision  :", receipt["decision"], "/", receipt["answer_state"])
print("cited rows:", len(receipt["cited_decision_ids"]))
print("evaluated :", receipt["provenance"]["evaluated_at"])


Same record, four verified identities, four access shapes —
each one a durable, citable decision. The role is never a
caller claim: it rides the token, and the provenance on every
answer states which identity was evaluated.
